# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [1]:
# TASK TYPE: Ranking / Scoring
#
# This is a ranking problem — not pure classification.
# The goal is NOT just "is this page declining? yes/no"
# The goal IS "which pages should an editor look at FIRST?"
#
# That means the output is a RANKED LIST ordered by priority score,
# not just a binary label. Classification is the engine under the hood
# (predict probability of decline), but the real deliverable is the ranking.
#
# From the skill framework:
#   "Which ones first?" → Ranking/scoring → priority score → precision@K

print("Task type: Ranking / Scoring")
print("Engine: binary classification (declining yes/no) to produce probabilities")
print("Deliverable: a ranked queue ordered by decline probability")
print("Why ranking, not just classification:")
print("  - Editors can only review N pages per week")
print("  - They need the WORST ones first, not just a yes/no flag")
print("  - Precision@K measures exactly this: of the top K picks, how many are real?")

Task type: Ranking / Scoring
Engine: binary classification (declining yes/no) to produce probabilities
Deliverable: a ranked queue ordered by decline probability
Why ranking, not just classification:
  - Editors can only review N pages per week
  - They need the WORST ones first, not just a yes/no flag
  - Precision@K measures exactly this: of the top K picks, how many are real?


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [2]:
# TARGET: is_declining_label = (trend_direction == "down")
#
# This is a PROXY label, not a perfect target. Here's why:
#
# - It comes from trend_direction, which is a bucket computed from trend_pct
#   in the CURRENT window — it's an observed outcome, but it describes the
#   present, not the future
#
# - A stronger capstone target would be:
#   "features from prior 90 days → decline in NEXT 30 days"
#   (requires the full warehouse release with time windows)
#
# - For now, this proxy is honest enough to start: it IS measured from real
#   search data, and the starter pipeline proved there's learnable signal
#
# LEAKAGE RULES:
# - trend_direction and trend_pct are NEVER features (they ARE the label)
# - IDs (content_id, client_id) are for grouping only, never features

print("Target: is_declining_label = (trend_direction == 'down')")
print("Type: binary proxy label (observed outcome, current window)")
print("Source: derived from trend_direction in the starter CSV")
print()
print("Leakage-banned columns:")
print("  - trend_direction (the label itself)")
print("  - trend_pct (the number the label was computed from)")
print()
print("Future improvement: build a forward-looking label from the warehouse")
print("  features: prior 90 days → target: decline in next 30 days")

Target: is_declining_label = (trend_direction == 'down')
Type: binary proxy label (observed outcome, current window)
Source: derived from trend_direction in the starter CSV

Leakage-banned columns:
  - trend_direction (the label itself)
  - trend_pct (the number the label was computed from)

Future improvement: build a forward-looking label from the warehouse
  features: prior 90 days → target: decline in next 30 days


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [3]:
# SUCCESS METRIC: Precision@50
#
# Why Precision@50:
# - It matches the real decision: "of the top 50 pages we flag for review,
#   how many are actually declining?"
# - An editor's time is the scarce resource — every false positive is wasted time
# - The starter baseline got Precision@50 = 0.240 (12 of 50 right)
# - The starter model got Precision@50 = 0.740 (37 of 50 right)
# - "Good" = clearly beating 0.240 on the same split
#
# Supporting metrics (not the primary goal, but useful context):
# - ROC-AUC: overall discrimination ability
# - Average Precision: quality of the full ranking, not just top 50
# - Base rate: the decline rate (~47%) — any metric must beat random guessing

print("Primary metric: Precision@50")
print("  'Of the top 50 pages flagged, how many are actually declining?'")
print()
print("Baseline to beat: 0.240 (hand-written rule)")
print("Starter model:    0.740 (random forest)")
print("Base rate:        ~0.47  (random guessing)")
print()
print("Supporting metrics: ROC-AUC, Average Precision")
print("Always report base rate next to any precision number")

Primary metric: Precision@50
  'Of the top 50 pages flagged, how many are actually declining?'

Baseline to beat: 0.240 (hand-written rule)
Starter model:    0.740 (random forest)
Base rate:        ~0.47  (random guessing)

Supporting metrics: ROC-AUC, Average Precision
Always report base rate next to any precision number


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ak8x6/flyrank-seo-ml-pipeline"
REPO_DIR = "flyrank-seo-ml-pipeline"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.join("..", ".."))

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The unit of analysis: ONE ROW = ONE CONTENT PAGE
# Each row is a unique pseudonymized content item with its trailing-90-day metrics
print(f"Unit of analysis: one row = one content page")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Unique content items: {df['content_id'].nunique():,}")
print(f"Unique clients: {df['client_id'].nunique()}")
print()

# Build the label
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Show a sample of key columns with the target
key_cols = ["content_id", "client_id", "impressions_90d", "avg_position",
            "ctr", "days_since_last_update", "content_age_days",
            "word_count", "trend_direction", "is_declining_label"]

print("Sample rows (key columns + target):")
df[key_cols].head(8)

Unit of analysis: one row = one content page
Shape: 30,000 rows x 44 columns
Unique content items: 30,000
Unique clients: 32

Sample rows (key columns + target):


,content_id,client_id,impressions_90d,avg_position,ctr,days_since_last_update,content_age_days,word_count,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,10.6,0.76,20,187,3221.0,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,0.05,25,445,2481.0,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,0.09,20,141,3515.0,down,1
3,content_331d6c4de07b,client_19581e27de,11751,6.2,0.49,22,463,NaN,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,0.13,14,263,2803.0,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,8.5,0.03,20,147,3080.0,down,1
6,content_9a34b442b552,client_8722616204,20,7.0,0.00,20,90,3059.0,down,1
7,content_a63219c6e95a,client_19581e27de,1724,21.2,0.06,22,445,NaN,stable,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [5]:
# WHY ML BEATS A FIXED RULE:
#
# The starter pipeline already proved this with real numbers:
# - Hand rule Precision@50: 0.240 → only 12 of 50 picks were right
# - Random forest Precision@50: 0.740 → 37 of 50 picks were right
#
# But WHY does the rule fail? Because:

# 1. Too many signals interact at once
print("Reason 1: Too many interacting signals")
print(f"  The dataset has {df.shape[1]} columns of observable signals")
print(f"  A hand rule uses 2-3 at most (e.g., stale + visible)")
print(f"  The model weighs all of them together")
print()

# 2. Non-obvious patterns
corr_age = df["content_age_days"].corr(df["is_declining_label"])
corr_wc = df["word_count"].corr(df["is_declining_label"])
print(f"Reason 2: Obvious features don't work alone")
print(f"  content_age_days vs decline correlation: {corr_age:.3f} (weak)")
print(f"  word_count vs decline correlation: {corr_wc:.3f} (near zero)")
print(f"  No single column cleanly separates declining from healthy pages")
print()

# 3. The pattern varies by client/content type
decline_by_type = df.groupby("content_type")["is_declining_label"].mean()
print(f"Reason 3: Decline rate varies by content type")
print(decline_by_type.round(3).to_string())
print()
print("A fixed rule applies the same thresholds everywhere.")
print("A model can learn that different content types decline for different reasons.")
print("That's why ML earns its place here — the pattern is real but too messy for an if-statement.")

Reason 1: Too many interacting signals
  The dataset has 45 columns of observable signals
  A hand rule uses 2-3 at most (e.g., stale + visible)
  The model weighs all of them together

Reason 2: Obvious features don't work alone
  content_age_days vs decline correlation: -0.164 (weak)
  word_count vs decline correlation: 0.090 (near zero)
  No single column cleanly separates declining from healthy pages

Reason 3: Decline rate varies by content type
content_type
comparison article    0.572
feedly article        0.287
keyword article       0.561

A fixed rule applies the same thresholds everywhere.
A model can learn that different content types decline for different reasons.
That's why ML earns its place here — the pattern is real but too messy for an if-statement.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.